In [9]:
import numpy as np
import json
import datasets

# read from ./amazon_data/CDs_and_Vinyl_train_sampled.json
with open('./amazon_data/CDs_and_Vinyl_train_sampled_orig.json', 'r') as f:
    sampled_data = json.load(f)


reasoning_instruction = """You should begin by analyzing and reasoning about the current user's viewing history to infer their preferences. To strengthen your reasoning, you may query a database that contains: (1) Interaction histories of other users; (2) Metadata of the musics including Price, SalesRank, Brand, and Categories.
Especially, you may identify the other users who have engaged with musics similar to those viewed by the current user. Then you can query and retrieve their interaction histories to discover additional musics they enjoyed. Then you can use these patterns to predict and recommend musics that the current user is also likely to appreciate.
Also, if further information is needed about a specific music, such as Price, SalesRank, Brand, or Categories, you may query the database to enrich the recommendation context.
"""


for data in sampled_data:
    question = data["instruction"] + "\n" + data["input"] + "\n" + reasoning_instruction
    prefix = f"""Resolve the given task. \
You must conduct reasoning inside <think> and </think> first every time you get new information. \
After reasoning, if you find you lack some knowledge, you can call a search engine by <search> query </search> and it will return the top searched results between <information> and </information>. \
You can search as many times as your want. \
If you find no further external knowledge needed, you can directly provide the answer inside <answer> and </answer> and using two double quotes to enclose the name of the music, without detailed illustrations. For example, <answer> "Revenge" </answer>. \Task: {question}\n"""
    data["question"] = prefix
    
# save the sampled_data to a dataset object
data_list = []
for data in sampled_data:
    data_list.append({
        "question": data["question"],
        "golden_answers": data["output"]
    })
train_dataset = datasets.Dataset.from_list(data_list)


In [10]:


import re
import os
import datasets



data_source = "amazon"
split = "train"


# add a row to each data item that represents a unique id
def make_map_fn(split):

    def process_fn(example, idx):
        question = example['question']
        solution = {
            "target": example['golden_answers'],
        }

        data = {
            "data_source": data_source,
            "prompt": [{
                "role": "user",
                "content": question,
            }],
            "ability": "fact-reasoning",
            "reward_model": {
                "style": "rule",
                "ground_truth": solution
            },
            "extra_info": {
                'split': split,
                'index': idx,
            }
        }
        return data

    return process_fn

train_dataset = train_dataset.map(function=make_map_fn('train'), with_indices=True)


train_dataset.to_parquet(os.path.join("./amazon_data", 'train.parquet'))




Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00, 251.99ba/s]


15787504

Test

In [6]:
import numpy as np
import json
import datasets

# read from ./amazon_data/CDs_and_Vinyl_train_sampled.json
with open('./amazon_data/CDs_and_Vinyl_test.json', 'r') as f:
    test_data = json.load(f)


reasoning_instruction = """You should begin by analyzing and reasoning about the current user's viewing history to infer their preferences. To strengthen your reasoning, you may query a database that contains: (1) Interaction histories of other users; (2) Metadata of the musics including Price, SalesRank, Brand, and Categories.
Especially, you may identify the other users who have engaged with musics similar to those viewed by the current user. Then you can query and retrieve their interaction histories to discover additional musics they enjoyed. Then you can use these patterns to predict and recommend musics that the current user is also likely to appreciate.
Also, if further information is needed about a specific music, such as Price, SalesRank, Brand, or Categories, you may query the database to enrich the recommendation context.
"""


# randomly sample 1000 data from test_data
np.random.seed(42)
test_data = np.random.choice(test_data, size=1000, replace=False)

for data in test_data:
    question = data["instruction"] + "\n" + data["input"] + "\n" + reasoning_instruction
    prefix = f"""Resolve the given task. \
You must conduct reasoning inside <think> and </think> first every time you get new information. \
After reasoning, if you find you lack some knowledge, you can call a search engine by <search> query </search> and it will return the top searched results between <information> and </information>. \
You can search as many times as your want. \
If you find no further external knowledge needed, you can directly provide the answer inside <answer> and </answer> and using two double quotes to enclose the name of the music, without detailed illustrations. For example, <answer> "Revenge" </answer>. \Task: {question}\n"""
    data["question"] = prefix
    
# save the test_data to a dataset object
data_list = []
for data in test_data:
    data_list.append({
        "question": data["question"],
        "golden_answers": data["output"]
    })
test_dataset = datasets.Dataset.from_list(data_list)

print("length of test dataset:", len(test_dataset))


length of test dataset: 1000


In [7]:


import re
import os
import datasets



data_source = "amazon"
split = "test"


# add a row to each data item that represents a unique id
def make_map_fn(split):

    def process_fn(example, idx):
        question = example['question']
        solution = {
            "target": example['golden_answers'],
        }

        data = {
            "data_source": data_source,
            "prompt": [{
                "role": "user",
                "content": question,
            }],
            "ability": "fact-reasoning",
            "reward_model": {
                "style": "rule",
                "ground_truth": solution
            },
            "extra_info": {
                'split': split,
                'index': idx,
            }
        }
        return data

    return process_fn

test_dataset = test_dataset.map(function=make_map_fn('test'), with_indices=True)


test_dataset.to_parquet(os.path.join("./amazon_data", 'test.parquet'))




Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 168.51ba/s]


3861388